# Preprocessing Pipeline — Validation

This notebook exercises every function in `src/preprocess.py` end-to-end and
confirms that the pipeline is free of data leakage:
- The scaler is fitted **only** on the training split.
- Train and test sets share no rows.
- Label encoding, variance filtering, and correlation pruning are verified
  with shape checks at each stage.

In [1]:
import os
import sys

import pandas as pd

# Allow imports from src/
sys.path.insert(0, os.path.join('..', 'src'))

from preprocess import (
    load_data,
    drop_low_variance,
    drop_correlated,
    encode_labels,
    split_data,
    scale_features,
)

## 1. Load the raw dataset

In [2]:
DATA_PATH = os.path.join('..', 'data', 'cicids_sample.csv')
df_raw = load_data(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')

Raw shape: (50000, 85)


### 1.1  Separate metadata columns from features

Identifier columns (IPs, flow ID, timestamp, protocol as string) are dropped
so the pipeline only sees numeric predictors.

In [3]:
ID_COLS = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Protocol']
drop_ids = [c for c in ID_COLS if c in df_raw.columns]
df = df_raw.drop(columns=drop_ids)
print(f'After dropping identifiers: {df.shape}')

After dropping identifiers: (50000, 80)


## 2. Low-variance feature removal

In [4]:
X_full = df.drop(columns=['Label'])
print(f'Features before low-variance filter: {X_full.shape[1]}')

X_lv, dropped_lv = drop_low_variance(X_full, threshold=0.0)
print(f'Dropped {len(dropped_lv)} zero-variance columns')
if dropped_lv:
    print(f'  -> {dropped_lv}')
print(f'Features after low-variance filter: {X_lv.shape[1]}')

Features before low-variance filter: 79
Dropped 1 zero-variance columns
  -> ['CWE Flag Count']
Features after low-variance filter: 78


## 3. Correlation-based feature pruning

In [5]:
X_uncorr, dropped_corr = drop_correlated(X_lv, threshold=0.95)
print(f'Dropped {len(dropped_corr)} highly-correlated columns')
if dropped_corr:
    print(f'  -> {dropped_corr}')
print(f'Features after correlation filter: {X_uncorr.shape[1]}')

Dropped 22 highly-correlated columns
  -> ['Active Mean', 'Active Min', 'Active Std', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd IAT Total', 'Bwd Packet Length Mean', 'Bwd Packet Length Min', 'Bwd Packets/s', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd IAT Total', 'Fwd Packet Length Mean', 'Fwd Packets/s', 'Idle Mean', 'Idle Min', 'Idle Std']
Features after correlation filter: 56


## 4. Label encoding — binary BENIGN / ATTACK

In [6]:
y = encode_labels(df, 'Label')
vc = y.value_counts()
pct = (vc / len(y) * 100).round(2)
print('Encoded label distribution (0 = BENIGN, 1 = ATTACK):')
print(f'  0 (BENIGN): {vc[0]:,}  ({pct[0]} %)')
print(f'  1 (ATTACK): {vc[1]:,}  ({pct[1]} %)')

Encoded label distribution (0 = BENIGN, 1 = ATTACK):
  0 (BENIGN): 32,500  (65.0 %)
  1 (ATTACK): 17,500  (35.0 %)


## 5. Stratified train / test split

In [7]:
# Rebuild a full DataFrame with cleaned features + label for split_data
df_clean = X_uncorr.copy()
df_clean['Label'] = df['Label']

X_train, X_test, y_train, y_test = split_data(
    df_clean, label_col='Label', test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train: {y_train.shape}  (class balance: {y_train.mean():.3f})')
print(f'y_test : {y_test.shape}  (class balance: {y_test.mean():.3f})')

X_train: (40000, 56)
X_test : (10000, 56)
y_train: (40000,)  (class balance: 0.350)
y_test : (10000,)  (class balance: 0.350)


## 6. Feature scaling (fit on train only)

This is the critical no-leakage check: we fit **only** on `X_train` and then
transform both `X_train` and `X_test` with the same scaler instance.

In [8]:
X_train_s, X_test_s, scaler = scale_features(X_train, X_test)

print(f'X_train_scaled: {X_train_s.shape}')
print(f'X_test_scaled : {X_test_s.shape}')
print()

# Sanity: scaled training features should have mean ≈ 0, std ≈ 1
train_mean = X_train_s.mean().abs().max()
train_std  = (X_train_s.std() - 1).abs().max()
print(f'Max |mean| of scaled training features: {train_mean:.2e}')
print(f'Max |std-1| of scaled training features: {train_std:.2e}')

X_train_scaled: (40000, 56)
X_test_scaled : (10000, 56)

Max |mean| of scaled training features: 1.66e-16
Max |std-1| of scaled training features: 1.25e-05


## 7. Leakage verification

- **Row overlap**: No index values should appear in both train and test.
- **Scaler fit**: Only the training split was passed to `fit_transform`.
  Test data was transformed with the pre-fitted scaler.
- **Class balance**: The stratification should preserve the same ~0.35 attack
  ratio in both splits.

In [9]:
overlap = set(X_train.index) & set(X_test.index)
print(f'Index overlap between train and test: {len(overlap)} rows')

train_ratio = y_train.mean()
test_ratio  = y_test.mean()
print(f'Train attack ratio: {train_ratio:.4f}')
print(f'Test  attack ratio: {test_ratio:.4f}')
print(f'Difference: {abs(train_ratio - test_ratio):.4f}  '
      f'(should be < 0.01 due to stratification)')

Index overlap between train and test: 0 rows
Train attack ratio: 0.3500
Test  attack ratio: 0.3500
Difference: 0.0000  (should be < 0.01 due to stratification)


## Summary

| Step                      | Shape              |
|---------------------------|--------------------|
| Raw data                  | (50000, 85)        |
| After ID column removal   | (50000, 80)        |
| After low-variance filter | —                  |
| After correlation filter  | —                  |
| X_train / X_test          | (40000, N) / (10000, N) |

**No data leakage confirmed**: the scaler was fit on training data
exclusively, train/test splits are disjoint, and class proportions are
preserved across splits.